# Run the metaphor search yourself — on public data, or on your own texts

This notebook **runs the pipeline** that the walkthrough only explains. Out of the box it
runs on a public corpus (the CC-licensed #ReframeCovid collection, with the 17 published
Metaphor Menu entries hidden inside as check items) and needs **no GPU at all** in
`DRYRUN` mode: a stand-in answers every model question, so you can watch the plumbing
work in about a minute. Switch `MODE = 'LIVE'` and point it at an Ollama server to use
real models — on the public corpus first, then on your own texts.

**What happens, in order** (each step writes plain files into `WORK`, so you can stop,
look, and resume):

1. **Cut** the texts into short passages.
2. **Two model families** read every passage and propose candidate metaphors.
3. **Agreement**: keep only what both found.
4. **Four screens**: is it really a metaphor about the illness experience? is it about
   *living with* it? is it a fresh image or a stock phrase? how much does it resemble the
   published Menu (0–10)?
5. **Rank** — and check where the hidden Menu entries landed.
6. **Build the review pages** (the blinded stage-1/2/3 pages) and the source-domain layers.

**One honest caveat about the public corpus.** The project's real English run searched 600
raw Reddit posts, most of which contain no menu-worthy metaphor at all — that needle-in-a-
haystack search is what the ranking was evaluated on (see `ranker_walkthrough.ipynb` and
the demo pages). Those posts cannot be redistributed, so this notebook's public corpus is
made of texts that were *submitted because* they contain a metaphor. It exercises every
step — mining, agreement, screening, ranking, pages — but the hunting part only shows
its worth on your own raw texts.

All heavy code lives in `pipeline/` next to this notebook; the cells below only call it.

## 1 · Configure

Change nothing to try it. For real models set `MODE = 'LIVE'` and `OLLAMA_URL`; the two
mining families and the screening models are the ones the project used — any two
*different* families will do (that is the point of the agreement step). For your own
data set `MY_TEXTS` to a folder of `.txt` files (one document per file) and `LANGUAGE`.

Everything stays on the machine that runs this notebook; no cloud service is called.

In [ ]:
from pathlib import Path
import json, sys, subprocess
sys.path.insert(0, str(Path.cwd()))
from pipeline import llm, segment, mine, agree, screens, rank, report, demo_corpus

MODE       = 'DRYRUN'                      # 'DRYRUN' (no model, runs anywhere) or 'LIVE' (Ollama)
OLLAMA_URL = 'http://localhost:11434'      # your Ollama server (LIVE only)
LANGUAGE   = 'English'                     # 'English', 'Danish' or 'Dutch' — picks the prompts

MINING_MODELS = ['qwen3:32b', 'gemma4:31b']            # two different families
SCREEN_MODELS = {'verify': 'qwen3:32b', 'register': 'gemma4:31b',
                 'experiential': 'gpt-oss:120b', 'score': 'gpt-oss:120b'}   # gpt-oss -> gemma4:31b if you have no 120B
WORKERS    = 8                             # parallel requests to Ollama (LIVE)
LIMIT      = None                          # e.g. 40 to mine only the first 40 passages (smoke test)

MY_TEXTS   = None                          # e.g. Path('/data/my_interviews')  -> folder of .txt files
MIN_WORDS  = 15                            # 25 for long interviews/posts, 8 for short questionnaire answers
WORK       = Path('work') / ('demo_' + LANGUAGE.lower() if MY_TEXTS is None else 'my_run')

import os                                   # optional overrides for unattended runs (nbconvert on a server)
MODE = os.environ.get('METAPHOR_MODE', MODE); OLLAMA_URL = os.environ.get('OLLAMA_URL', OLLAMA_URL)
if os.environ.get('METAPHOR_SCREEN_FALLBACK'):      # e.g. 'gemma4:31b' when no gpt-oss:120b is available
    SCREEN_MODELS = {k: (os.environ['METAPHOR_SCREEN_FALLBACK'] if v.startswith('gpt-oss') else v) for k, v in SCREEN_MODELS.items()}

client = llm.client('dryrun' if MODE == 'DRYRUN' else OLLAMA_URL)
WORK.mkdir(parents=True, exist_ok=True)
if MODE == 'LIVE':
    have = client.models()
    need = set(MINING_MODELS) | set(SCREEN_MODELS.values())
    missing = [m for m in need if not any(h.startswith(m) for h in have)]
    print('Ollama models available:', have)
    assert not missing, f'pull these models first: ollama pull {" ".join(missing)}'
print(f'{MODE} run | language {LANGUAGE} | work dir {WORK.resolve()}')

## 2 · The texts, cut into passages

With `MY_TEXTS = None` this loads the public corpus: #ReframeCovid entries in your
language, and — for English — the 17 Menu entries planted as known-good items (their
document ids start with `MENU_`, which is how the ranking recognises them later).

In [ ]:
if MY_TEXTS is None:
    texts, planted = demo_corpus.build(LANGUAGE)
    print(f'public corpus: {len(texts) - len(planted)} #ReframeCovid entries + {len(planted)} planted Menu entries')
    min_words = 6                                   # ReframeCovid entries are short
else:
    texts = {p.stem: p.read_text(encoding='utf-8', errors='ignore') for p in sorted(Path(MY_TEXTS).glob('*.txt'))}
    planted = []
    print(f'{len(texts)} documents from {MY_TEXTS}')
    min_words = MIN_WORDS

segs = segment.from_texts(texts, window=3, min_words=min_words)
print(segment.save(segs, WORK / 'segments.json'))
print('\nexample passage:', repr(segs[0][1][:160]))

## 3 · Two model families read every passage

Each model gets the same open question ("find all metaphorical expressions") and answers
in JSON. Answers are checkpointed per model, so a stopped run resumes where it was.
In `DRYRUN` the stand-in invents candidates from the passage text — meaningless, but the
files have the right shape.

In [ ]:
for m in MINING_MODELS:
    print(mine.run(client, m, WORK, language=LANGUAGE, workers=WORKERS, limit=LIMIT))

## 4 · Keep only what both families found

A candidate survives when the *other* family proposed an overlapping phrase in the same
passage. This single step removed about two thirds of the raw proposals in the project
while keeping the known-good items.

In [ ]:
print(agree.run(WORK, MINING_MODELS))

## 5 · Four screens

Each screen is one yes/no (or 0–10) question per candidate, asked of a local model. The
two expensive screens (experiential, score) run only on candidates that passed the strict
verify screen, plus the planted items.

In [ ]:
for r in screens.run_all(client, WORK, SCREEN_MODELS, language=LANGUAGE, workers=WORKERS):
    print(r)

## 6 · Rank, and check the hidden items

The order is: passed verify *and* experiential first, then by Menu-likeness score, vivid
before conventional. Then the question that matters: **where did the planted Menu entries
land?** In the project's English evaluation their median rank was 107 of 1,490 (10 of 13
in the top 10%). On this small public corpus with real models expect them near the top;
in `DRYRUN` the positions are random by construction.

In [ ]:
print(rank.run(WORK, SCREEN_MODELS))
print()
summary = report.summary(WORK)

## 7 · The review pages and the source-domain layers

`review_page.py` builds the same three pages used in the project: **stage 1** (random
order, for researchers to filter), **stage 2** (random order, for the PPI panel to vote)
and **stage 3** (the ranked working copy with every verdict, the source-domain table and
the search box). The source-domain layers (head word, WordNet, and — LIVE only — a short
concept named by a local model, clustered by embedding similarity) come from
`vehicle_layers.py`; they need `spacy`, `nltk` and `sentence-transformers` installed.
Open `index.html` in the work folder when the cell finishes.

In [ ]:
PIPE = Path.cwd() / 'pipeline'
corpus_name = 'Public demo' if MY_TEXTS is None else 'My corpus'
import shutil
shutil.copy(PIPE / 'demo.css', WORK / 'demo.css')          # the shared presentation layer
theme = ['--demo-theme'] + ([] if MY_TEXTS is None else ['--theme-private']) + ['--theme-tag', f'{corpus_name} — {LANGUAGE}']
common = ['--stack', str(WORK), '--ranking', 'ranking.json', '--plant-prefix', 'PLANT',
          '--corpus', corpus_name, '--stratum-noun', 'source', '--top', '100000', *theme,
          '--source-note', f'{corpus_name} — {LANGUAGE}',
          '--provenance', f'{len(texts)} documents || {len(segs)} passages || two model families, agreement, four screens || ranked']

# optional layers (skipped if the libraries are not installed)
lay = WORK / 'vehicle_layers.jsonl'
try:
    import spacy, nltk  # noqa
    lang_code = {'English': 'en', 'Danish': 'da', 'Dutch': 'nl'}[LANGUAGE]
    subprocess.run([sys.executable, str(PIPE / 'vehicle_layers.py'), 'lexical', '--jobs', str(WORK / 'screens/jobs.json'),
                    '--language', lang_code, '--out', str(lay)], check=True)
    if MODE == 'LIVE':
        subprocess.run([sys.executable, str(PIPE / 'vehicle_layers.py'), 'llm', '--jobs', str(WORK / 'screens/jobs.json'),
                        '--language', lang_code, '--inout', str(lay), '--model', MINING_MODELS[0], '--host', OLLAMA_URL], check=True)
        for th, field in ((0.25, 'llm_c1'), (0.45, 'llm_c2')):
            subprocess.run([sys.executable, str(PIPE / 'vehicle_layers.py'), 'cluster', '--inout', str(lay),
                            '--threshold', str(th), '--field', field, '--show', '0'], check=True)
    common += ['--vehicle-layers', str(lay)]
except Exception as e:
    print('layers skipped:', e)

for stage, extra in (('explore', []), ('filter', ['--filter-top', '120', '--filter-deep', '40']), ('vote', ['--shortlist', ''])):
    if stage == 'vote':   # stage 2 needs a researcher shortlist; demo with the top tier
        ranking = json.loads((WORK / 'ranking.json').read_text())
        (WORK / 'shortlist.json').write_text(json.dumps([r['id'] for r in ranking if r['tier'] == 0][:60]))
        extra = ['--shortlist', str(WORK / 'shortlist.json')]
    out = subprocess.run([sys.executable, str(PIPE / 'review_page.py')] + common + ['--stage', stage] + extra,
                         capture_output=True, text=True)
    import re
    m = re.search(r'"path":\s*"([^"]+)"', out.stdout)
    print(stage, '->', (m.group(1) if (out.returncode == 0 and m) else out.stderr[-600:]))
subprocess.run([sys.executable, str(PIPE / 'review_index.py'), '--dir', str(WORK), '--title', f'{corpus_name} — review workflow',
                '--demo-theme', '--theme-tag', f'{corpus_name} — {LANGUAGE}',
                '--data', f'{corpus_name}, {LANGUAGE}. Built by run_on_your_data.ipynb.'], check=True)
print('\nopen:', (WORK / 'index.html').resolve())

## Using this on private data — the rules the project worked by

- **Local only.** Set `OLLAMA_URL` to a machine that holds the data; never a hosted API.
  The project's privacy gate refused any model name that was not a local one.
- **Numbers leave, text does not.** `report.summary` prints counts and ranks only. The
  review pages embed passages, so they are as private as the corpus: keep them on project
  machines and send reviewers files, not links.
- **Plant your check items.** Put a few known-good metaphors into the corpus as documents
  whose id starts with `MENU_` (translate the published Menu if needed). If they do not
  surface near the top, do not trust the ranking on that corpus.
- **Expect the register to matter.** Short direct answers (questionnaires) produced Menu
  candidates about twenty times as often as interviews; use a lower `MIN_WORDS` for them.